# Tensor Titans: Improved Multi-Horizon Wildfire Survival Baseline

This notebook replaces the old binary-event baseline with a reproducible multi-horizon workflow that:

- handles right-censoring separately at 12h, 24h, 48h, and 72h;
- evaluates predictions out-of-fold using C-index and censor-aware Brier scores;
- blends a regularized linear model with a small nonlinear model;
- enforces monotonic probabilities across time horizons; and
- creates a submission matching `sample_submission.csv`.

The original notebook is preserved. Run this notebook from top to bottom in Google Colab or a Python environment with pandas, NumPy, scikit-learn, matplotlib, and seaborn.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
HORIZONS = np.array([12, 24, 48, 72])
HORIZON_COLUMNS = [f'prob_{h}h' for h in HORIZONS]
np.random.seed(RANDOM_STATE)


## 1. Load and inspect the data

`event=1` means the fire reached the evacuation-zone threshold. `time_to_hit_hours` is either the observed event time or the last observed time for a censored fire.

In [ ]:
DATA_DIR = Path('.')
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
metadata = pd.read_csv(DATA_DIR / 'metaData.csv')

TARGET_COLUMNS = ['time_to_hit_hours', 'event']
ID_COLUMN = 'event_id'
FEATURE_COLUMNS = [c for c in test.columns if c != ID_COLUMN]

assert set(FEATURE_COLUMNS).issubset(train.columns)
assert list(sample_submission.columns) == [ID_COLUMN] + HORIZON_COLUMNS

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Features:    {len(FEATURE_COLUMNS)}')
print(f'Observed events: {int(train.event.sum())}/{len(train)} ({train.event.mean():.1%})')
display(train.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=train, x='event', ax=axes[0])
axes[0].set_title('Observed event vs censored')
sns.histplot(data=train, x='time_to_hit_hours', hue='event', bins=20, multiple='stack', ax=axes[1])
axes[1].set_title('Observed/censoring times')
plt.tight_layout()

missing = train[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False)
print('Features containing missing values:', int((missing > 0).sum()))
display(missing[missing > 0].to_frame('missing_count').head(10))


## 2. Censor-aware labels and competition-style metrics

At horizon `h`, an example is positive when an observed event occurred by `h`. It is a known negative when observation continued beyond `h`. A censored example whose observation ended on or before `h` is excluded because its status at `h` is unknown.

In [ ]:
def horizon_labels(times, events, horizon):
    times = np.asarray(times)
    events = np.asarray(events).astype(bool)
    # At the terminal horizon, every non-event is a known negative: the
    # competition defines it as not hitting within the full 72h window.
    known = np.ones(len(times), dtype=bool) if horizon == HORIZONS[-1] else (events | (times > horizon))
    labels = (events & (times <= horizon)).astype(int)
    return labels, known


def censor_aware_brier(times, events, probabilities, horizon):
    labels, known = horizon_labels(times, events, horizon)
    return np.mean((labels[known] - probabilities[known]) ** 2)


def concordance_index(times, events, risk):
    times = np.asarray(times)
    events = np.asarray(events).astype(bool)
    risk = np.asarray(risk)
    concordant = comparable = 0.0
    for i in range(len(times)):
        if not events[i]:
            continue
        later = np.flatnonzero(times > times[i])
        comparable += len(later)
        concordant += np.sum(risk[i] > risk[later])
        concordant += 0.5 * np.sum(risk[i] == risk[later])
    return concordant / comparable if comparable else np.nan


def evaluate_predictions(times, events, probabilities, verbose=True):
    probabilities = np.asarray(probabilities)
    brier = {
        h: censor_aware_brier(times, events, probabilities[:, i], h)
        for i, h in enumerate(HORIZONS)
    }
    weighted_brier = 0.3 * brier[24] + 0.4 * brier[48] + 0.3 * brier[72]
    c_index = concordance_index(times, events, probabilities[:, -1])
    hybrid = 0.3 * c_index + 0.7 * (1 - weighted_brier)
    result = {'c_index': c_index, 'weighted_brier': weighted_brier, 'hybrid': hybrid, **{f'brier_{h}h': brier[h] for h in HORIZONS}}
    if verbose:
        display(pd.Series(result, name='score').to_frame())
    return result


label_summary = []
for h in HORIZONS:
    labels, known = horizon_labels(train.time_to_hit_hours, train.event, h)
    label_summary.append({'horizon': h, 'eligible': int(known.sum()), 'excluded_censored': int((~known).sum()), 'positives': int(labels[known].sum())})
display(pd.DataFrame(label_summary))


## 3. Models and out-of-fold validation

The logistic model gives stable, regularized probabilities on this small dataset. HistGradientBoosting adds limited nonlinear capacity. Their fixed blend avoids tuning weights against a tiny validation set.

In [ ]:
linear_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=0.2, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE)),
])

nonlinear_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('model', HistGradientBoostingClassifier(
        learning_rate=0.04,
        max_iter=150,
        max_leaf_nodes=7,
        min_samples_leaf=15,
        l2_regularization=2.0,
        random_state=RANDOM_STATE,
    )),
])

MODEL_WEIGHTS = (0.70, 0.30)


def fit_predict_blend(X_train, y_train, X_predict):
    linear = clone(linear_model).fit(X_train, y_train)
    nonlinear = clone(nonlinear_model).fit(X_train, y_train)
    return (
        MODEL_WEIGHTS[0] * linear.predict_proba(X_predict)[:, 1]
        + MODEL_WEIGHTS[1] * nonlinear.predict_proba(X_predict)[:, 1]
    )


X = train[FEATURE_COLUMNS]
times = train.time_to_hit_hours.to_numpy()
events = train.event.to_numpy()
oof_predictions = np.zeros((len(train), len(HORIZONS)))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, events), start=1):
    for horizon_idx, horizon in enumerate(HORIZONS):
        labels, known = horizon_labels(times, events, horizon)
        eligible_train_idx = train_idx[known[train_idx]]
        oof_predictions[valid_idx, horizon_idx] = fit_predict_blend(
            X.iloc[eligible_train_idx], labels[eligible_train_idx], X.iloc[valid_idx]
        )
    print(f'Completed fold {fold}')

# Cumulative event probabilities must never decrease as the horizon increases.
oof_predictions = np.maximum.accumulate(oof_predictions, axis=1)
oof_predictions = np.clip(oof_predictions, 0.001, 0.999)

oof_scores = evaluate_predictions(times, events, oof_predictions)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, h in enumerate(HORIZONS):
    sns.kdeplot(oof_predictions[:, i], label=f'{h}h', ax=axes[0], fill=False)
axes[0].set_title('Out-of-fold probability distributions')
axes[0].legend()

mean_by_event = pd.DataFrame(oof_predictions, columns=HORIZON_COLUMNS).assign(event=events).groupby('event').mean().T
mean_by_event.plot(kind='bar', ax=axes[1])
axes[1].set_title('Mean OOF probability by final event status')
axes[1].set_ylabel('Mean probability')
plt.tight_layout()


## 4. Train on all eligible data and create the submission

In [ ]:
X_test = test[FEATURE_COLUMNS]
test_predictions = np.zeros((len(test), len(HORIZONS)))

for horizon_idx, horizon in enumerate(HORIZONS):
    labels, known = horizon_labels(times, events, horizon)
    test_predictions[:, horizon_idx] = fit_predict_blend(X.loc[known], labels[known], X_test)
    print(f'Trained final {horizon}h ensemble on {known.sum()} eligible rows')

test_predictions = np.maximum.accumulate(test_predictions, axis=1)
test_predictions = np.clip(test_predictions, 0.001, 0.999)

submission = sample_submission.copy()
assert submission[ID_COLUMN].equals(test[ID_COLUMN])
submission[HORIZON_COLUMNS] = test_predictions

assert ((submission[HORIZON_COLUMNS] >= 0) & (submission[HORIZON_COLUMNS] <= 1)).all().all()
assert (np.diff(submission[HORIZON_COLUMNS].to_numpy(), axis=1) >= 0).all()
assert not submission.isna().any().any()

submission.to_csv('submission_improved.csv', index=False)
display(submission.head())
print('Saved submission_improved.csv')


## Next experiments

Use the out-of-fold hybrid score before accepting changes. Good next experiments include repeated cross-validation, probability calibration performed strictly within folds, survival-specific models such as Coxnet/Random Survival Forest, and a carefully validated feature subset. Avoid assigning one binary-event probability to every horizon because that discards event timing.